In [1]:
import numpy as np
import  pandas as pd
import librosa, librosa.display
import matplotlib.pyplot as plt
import os
import torch, torchaudio
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import IPython.display as ipd
import random
import seaborn as sns
from sklearn.model_selection import StratifiedShuffleSplit

# Data Loading Pipeline

In [2]:
BASE = "/kaggle/input/competitions/birdclef-2025"  # <-- CHANGE THIS to what FOUND printed
# example: BASE = "/kaggle/input/birdclef-2025/birdclef-2025" if needed

train = pd.read_csv(f"{BASE}/train.csv")
taxonomy = pd.read_csv(f"{BASE}/taxonomy.csv")
print(BASE, train.shape)

/kaggle/input/competitions/birdclef-2025 (28564, 13)


In [3]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
sample_df=train.sample(frac=0.3,random_state=42)
sample_df.shape

(8569, 13)

In [5]:
labels=sorted(train['primary_label'].unique())
label2idx={l:i for i,l in enumerate(labels)}
idx2label={i:l for l,i in label2idx.items()}
class CFG:
    SR=32000
    DURATION=5
    N_MELS=128
    BATCH=32
class BirdDataset(Dataset):
    def __init__(self, df):
        self.df = df
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=CFG.SR, n_mels=CFG.N_MELS, f_min=50, f_max=14000, n_fft=1024, hop_length=512
        )
        self.db = torchaudio.transforms.AmplitudeToDB()

    def __len__(self):
        return len(self.df)

    def __getitem__(self,idx):
        row=self.df.iloc[idx]
        path = os.path.join(BASE, "train_audio", row['filename'])
        wav,sr=torchaudio.load(path)
        wav=wav.mean(dim=0)
        if sr != CFG.SR:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=CFG.SR)
            wav = resampler(wav)
        target = CFG.SR * CFG.DURATION
        if len(wav) > target:
            start = torch.randint(0, len(wav)-target, (1,)).item()
            wav = wav[start:start+target]
        else:
            wav = F.pad(wav, (0, target - len(wav)))
            
        mel = self.mel(wav)
        mel = self.db(mel)
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)
        
        label = label2idx[row['primary_label']]
        return mel, label


In [6]:
splitter=StratifiedShuffleSplit(n_splits=1,test_size=0.2,random_state=42)
counts=sample_df['primary_label'].value_counts()
rare_classes=counts[counts==1].index
sample_df = sample_df[~sample_df['primary_label'].isin(rare_classes)].reset_index(drop=True)

In [7]:
sample_df.shape

(8555, 13)

In [8]:
train_idx, val_idx = next(splitter.split(sample_df, sample_df['primary_label']))

In [9]:
train_df = sample_df.iloc[train_idx].reset_index(drop=True)
val_df = sample_df.iloc[val_idx].reset_index(drop=True)

print(f"Train size: {len(train_df)} | Val size: {len(val_df)}")

Train size: 6844 | Val size: 1711


In [10]:
train_dataset=BirdDataset(train_df)
val_dataset=BirdDataset(val_df)

In [11]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

In [12]:
mels, labs = next(iter(train_loader))
print("Batch shape:", mels.shape)       
print("Labels shape:", labs.shape)

Batch shape: torch.Size([16, 128, 313])
Labels shape: torch.Size([16])


# Pure AST Model

In [13]:
from transformers import ASTForAudioClassification, ASTConfig

In [14]:
# Create the AST model manually with the 206 classes
config = ASTConfig.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
config.num_labels = 206
config.num_mel_bins = 128
config.max_length = 313 # Our time frames

model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    config=config,
    ignore_mismatched_sizes=True
).to(device)

# Sanity check
dummy = torch.randn(2, 128, 313).to(device) # Note: AST takes [B, 128, Time] without the channel dim!
with torch.no_grad():
    out = model(dummy).logits
print(out.shape) # Must be [2, 206]

config.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

[transformers] ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                                                          | Status   |                                                                                                  
-------------------------------------------------------------+----------+--------------------------------------------------------------------------------------------------
classifier.dense.bias                                        | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527]) vs model:torch.Size([206])                 
audio_spectrogram_transformer.embeddings.position_embeddings | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1, 1214, 768]) vs model:torch.Size([1, 362, 768])
classifier.dense.weight                                      | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527, 768]) vs model:torch.Size([206, 768])       

Notes:
- MISMATCH:	ckpt weights were loa

torch.Size([2, 206])


In [15]:
print(model)

ASTForAudioClassification(
  (audio_spectrogram_transformer): ASTModel(
    (embeddings): ASTEmbeddings(
      (patch_embeddings): ASTPatchEmbeddings(
        (projection): Conv2d(1, 768, kernel_size=(16, 16), stride=(10, 10))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0-11): 12 x ASTLayer(
        (attention): ASTAttention(
          (q_proj): Linear(in_features=768, out_features=768, bias=True)
          (k_proj): Linear(in_features=768, out_features=768, bias=True)
          (v_proj): Linear(in_features=768, out_features=768, bias=True)
          (o_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (layernorm_before): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (layernorm_after): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (mlp): ASTMLP(
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Li

In [16]:
# Total
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total/1e6:.1f}M | Trainable: {trainable/1e6:.1f}M")

# Per top-level module
for name, mod in model.named_children():
    n = sum(p.numel() for p in mod.parameters())
    print(f"{name:40s} {n/1e6:8.2f}M")

Total params: 85.7M | Trainable: 85.7M
audio_spectrogram_transformer               85.53M
classifier                                   0.16M


In [17]:
# for name, p in model.named_parameters():
#     print(f"{name:70s} {tuple(p.shape)}  requires_grad={p.requires_grad}")

In [18]:
with torch.no_grad():
    out = model(torch.randn(2, 128, 313).to(device))
print(type(out))            # AudioClassifierOutput
print(out.keys())           # ['logits']  (or ['logits','hidden_states',...])
print(out.logits.shape)     # torch.Size([2, 206])

<class 'transformers.modeling_outputs.SequenceClassifierOutput'>
odict_keys(['logits'])
torch.Size([2, 206])


In [19]:
# from transformers import AutoConfig, ASTForAudioClassification

# cfg = AutoConfig.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
# print(cfg)                                         # all hyperparams
# model = ASTForAudioClassification(cfg)             # uninitialized, same arch
# print(model)                                       # layer tree without downloading 346MB